In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("✓ Visualization libraries loaded")

In [1]:
# Check and install required libraries
import sys
import subprocess

def check_and_install(package):
    """Check if package is installed, if not, install it"""
    try:
        __import__(package)
        print(f"✓ {package} is already installed")
        return True
    except ImportError:
        print(f"✗ {package} is missing. Installing...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
            print(f"✓ {package} installed successfully")
            return True
        except subprocess.CalledProcessError as e:
            print(f"✗ Failed to install {package}: {e}")
            return False

# Required packages for Excel reading
required_packages = ['pandas', 'openpyxl']

print("Checking required packages...\n")
all_installed = True
for package in required_packages:
    if not check_and_install(package):
        all_installed = False

if all_installed:
    print("\n✅ All required packages are ready!")
else:
    print("\n⚠️ Some packages failed to install. Please install manually:")
    print("pip install pandas openpyxl")

Checking required packages...

✓ pandas is already installed
✗ openpyxl is missing. Installing...
✓ openpyxl installed successfully

✅ All required packages are ready!


# 📊 Data Exploration - Excel Data Analysis

This notebook explores the raw Excel data file and creates initial CSV files for further processing.

## What This Notebook Does:
1. Loads and explores all sheets in the Excel workbook
2. Joins product and demand history tables
3. Joins transaction tables with product information
4. Filters and prepares data for time series analysis

## Required Libraries
Run the next cell first to ensure all dependencies are installed.

In [2]:
import pandas as pd

# Updated path - notebook is now in ai/notebooks/, data is in ai/data/raw/
file_path = "../data/raw/WMS_Hackathon_DataPack_Templates_FR_FV_B7_ONLY.xlsx"

# Load workbook
xls = pd.ExcelFile(file_path)

for sheet in xls.sheet_names:
    print("\n" + "="*70)
    print(f"Sheet Name: {sheet}")
    print("="*70)

    try:
        df = pd.read_excel(xls, sheet_name=sheet)

        if df.empty:
            print("Sheet is empty.\n")
            continue

        print("\nColumns:")
        print(list(df.columns))

        print("\nFirst 5 rows:")
        print(df.head(5))

    except Exception as e:
        print(f"Error reading sheet {sheet}: {e}")


Sheet Name: LISEZ_MOI

Columns:
['WMS Hackathon - Modèles de pack de données (starter)']

First 5 rows:
  WMS Hackathon - Modèles de pack de données (starter)
0                                                NaN  
1                                   Comment utiliser  
2  1) Chaque onglet correspond à un modèle de fic...  
3  2) Ligne 1 = en-têtes de colonnes. Ligne 2 = t...  
4  3) Les équipes peuvent ajouter des colonnes, m...  

Sheet Name: DICTIONNAIRE_DONNEES

Columns:
['Table', 'Colonne', 'Type', 'Obligatoire', 'Description', 'Exemple']

First 5 rows:
      Table       Colonne   Type Obligatoire  \
0  produits    id_produit  texte           O   
1  produits           sku  texte           O   
2  produits   nom_produit  texte           O   
3  produits  unite_mesure  texte           O   
4  produits     categorie  texte           N   

                                         Description         Exemple  
0    Identifiant unique du produit (UUID ou entier).            P001  
1  Ré

---

## Step 1: Explore All Excel Sheets

Let's load the Excel file and see what data we have in each sheet.

---

## Step 2: Merge Products and Demand History

Join the `produits` table with `historique_demande` to create a complete product demand dataset.

In [3]:
# Join produits and historique_demand tables
import pandas as pd

# Updated path - notebook is now in ai/notebooks/, data is in ai/data/raw/
file_path = "../data/raw/WMS_Hackathon_DataPack_Templates_FR_FV_B7_ONLY.xlsx"

# Load the two sheets
df_produits = pd.read_excel(file_path, sheet_name='produits')
df_historique = pd.read_excel(file_path, sheet_name='historique_demande')

# Display the columns to verify the join key
print("Produits columns:", list(df_produits.columns))
print("Historique demand columns:", list(df_historique.columns))

# Join the tables on id_produit
# Adjust the column names if they're different
df_merged = pd.merge(df_produits, df_historique, on='id_produit', how='inner')

print(f"\nMerged table shape: {df_merged.shape}")
print("\nFirst 5 rows of merged data:")
print(df_merged.head())

# Save to CSV in data/raw directory
df_merged.to_csv('../data/raw/products_for_ts.csv', index=False)
print("\nData saved to ../data/raw/products_for_ts.csv")

Produits columns: ['id_produit', 'sku', 'nom_produit', 'unite_mesure', 'categorie', 'actif', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Poids(kg)', 'Is_Gerbable']
Historique demand columns: ['date', 'id_produit', 'quantite_demande']

Merged table shape: (151224, 13)

First 5 rows of merged data:
  id_produit  sku nom_produit unite_mesure categorie actif colisage fardeau  \
0      31334  NaN         NaN     Unité(s)   MOULURE  True               32   
1      31334  NaN         NaN     Unité(s)   MOULURE  True               32   
2      31334  NaN         NaN     Unité(s)   MOULURE  True               32   
3      31334  NaN         NaN     Unité(s)   MOULURE  True               32   
4      31334  NaN         NaN     Unité(s)   MOULURE  True               32   

  colisage palette volume pcs (m3) Poids(kg) Is_Gerbable                date  \
0             1600          0.0002         0        True 2024-04-21 08:31:24   
1             1600          0.0002         0       

In [ ]:
# 📊 VISUALIZE: Products & Demand Patterns
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Demand distribution
axes[0, 0].hist(df['quantite_demande'], bins=50, edgecolor='black', color='#4ECDC4', alpha=0.7)
axes[0, 0].set_xlabel('Demand Quantity', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Distribution of Demand Quantities', fontsize=12, fontweight='bold')
axes[0, 0].axvline(df['quantite_demande'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['quantite_demande'].mean():.0f}")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Top 15 products by total demand
top_products = df.groupby('nom_produit')['quantite_demande'].sum().nlargest(15).sort_values()
axes[0, 1].barh(range(len(top_products)), top_products.values, color='#FF6B6B', alpha=0.7)
axes[0, 1].set_yticks(range(len(top_products)))
axes[0, 1].set_yticklabels([name[:30] + '...' if len(name) > 30 else name for name in top_products.index], fontsize=9)
axes[0, 1].set_xlabel('Total Demand', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Top 15 Products by Total Demand', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Demand by category
if 'categorie' in df.columns:
    category_demand = df.groupby('categorie')['quantite_demande'].sum().sort_values(ascending=False)
    axes[1, 0].bar(range(len(category_demand)), category_demand.values, color='#95DAC1', alpha=0.7)
    axes[1, 0].set_xticks(range(len(category_demand)))
    axes[1, 0].set_xticklabels(category_demand.index, rotation=45, ha='right', fontsize=10)
    axes[1, 0].set_ylabel('Total Demand', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Demand by Product Category', fontsize=12, fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Number of products per category
if 'categorie' in df.columns:
    product_counts = df.groupby('categorie')['id_produit'].nunique().sort_values(ascending=False)
    colors_pie = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3']
    axes[1, 1].pie(product_counts.values, labels=product_counts.index, autopct='%1.1f%%', 
                   colors=colors_pie, startangle=90)
    axes[1, 1].set_title('Product Distribution by Category', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Summary Statistics:")
print(f"   Total demand records: {len(df):,}")
print(f"   Unique products: {df['id_produit'].nunique()}")
print(f"   Average demand per order: {df['quantite_demande'].mean():.2f}")
print(f"   Median demand: {df['quantite_demande'].median():.0f}")
print(f"   Max demand: {df['quantite_demande'].max():.0f}")

---

## Step 3: Join Transaction Tables

Join `transactions`, `lignes_transaction`, and `produits` tables to get complete transaction history with product details.

In [4]:
import pandas as pd

# Updated path - notebook is now in ai/notebooks/, data is in ai/data/raw/
file_path = "../data/raw/WMS_Hackathon_DataPack_Templates_FR_FV_B7_ONLY.xlsx"

# Load the three tables
df_transactions = pd.read_excel(file_path, sheet_name='transactions')
df_lignes_transaction = pd.read_excel(file_path, sheet_name='lignes_transaction')
df_produits = pd.read_excel(file_path, sheet_name='produits')

print("Transactions shape:", df_transactions.shape)
print("Transactions columns:", list(df_transactions.columns))

print("\nLignes transaction shape:", df_lignes_transaction.shape)
print("Lignes transaction columns:", list(df_lignes_transaction.columns))

print("\nProduits shape:", df_produits.shape)
print("Produits columns:", list(df_produits.columns))

# Step 1: Join transactions with lignes_transaction on id_transaction
print("\n" + "="*70)
print("Step 1: Joining transactions with lignes_transaction...")
print("="*70)
df_merged = pd.merge(df_transactions, df_lignes_transaction, on='id_transaction', how='inner')
print(f"After first join: {df_merged.shape}")

# Step 2: Join with produits on id_produit
print("\n" + "="*70)
print("Step 2: Joining with produits...")
print("="*70)
df_final = pd.merge(df_merged, df_produits, on='id_produit', how='inner')
print(f"Final shape: {df_final.shape}")

print("\nFinal columns:")
print(list(df_final.columns))

print("\nFirst 5 rows:")
print(df_final.head())

# Save to CSV in data/raw directory
df_final.to_csv('../data/raw/tempo_ts.csv', index=False)
print("\n✓ Data saved to ../data/raw/tempo_ts.csv")

Transactions shape: (4271, 7)
Transactions columns: ['id_transaction', 'type_transaction', 'reference_transaction', 'cree_le', 'cree_par_id_utilisateur', 'statut', 'notes']

Lignes transaction shape: (76890, 8)
Lignes transaction columns: ['id_transaction', 'no_ligne', 'id_produit', 'quantite', 'id_emplacement_source', 'id_emplacement_destination', 'lot_serie', 'code_motif']

Produits shape: (1583, 11)
Produits columns: ['id_produit', 'sku', 'nom_produit', 'unite_mesure', 'categorie', 'actif', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Poids(kg)', 'Is_Gerbable']

Step 1: Joining transactions with lignes_transaction...
After first join: (76890, 14)

Step 2: Joining with produits...
Final shape: (76890, 24)

Final columns:
['id_transaction', 'type_transaction', 'reference_transaction', 'cree_le', 'cree_par_id_utilisateur', 'statut', 'notes', 'no_ligne', 'id_produit', 'quantite', 'id_emplacement_source', 'id_emplacement_destination', 'lot_serie', 'code_motif', 'sku', 'nom

In [ ]:
# 📊 VISUALIZE: Transaction Tempo Data
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Convert date column to datetime for plotting
df_tempo_plot = df_tempo.copy()
if 'date' in df_tempo_plot.columns:
    df_tempo_plot['date'] = pd.to_datetime(df_tempo_plot['date'])
    
    # 1. Transactions over time
    daily_transactions = df_tempo_plot.groupby(df_tempo_plot['date'].dt.date).size()
    axes[0, 0].plot(daily_transactions.index, daily_transactions.values, marker='o', linewidth=2, markersize=4, color='#4ECDC4')
    axes[0, 0].set_xlabel('Date', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Number of Transactions', fontsize=11, fontweight='bold')
    axes[0, 0].set_title('Daily Transaction Volume', fontsize=12, fontweight='bold')
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Transaction types distribution
if 'type_transaction' in df_tempo_plot.columns:
    type_counts = df_tempo_plot['type_transaction'].value_counts()
    axes[0, 1].bar(range(len(type_counts)), type_counts.values, color=['#FF6B6B', '#4ECDC4', '#95DAC1', '#FECA57'][:len(type_counts)], alpha=0.7)
    axes[0, 1].set_xticks(range(len(type_counts)))
    axes[0, 1].set_xticklabels(type_counts.index, rotation=45, ha='right')
    axes[0, 1].set_ylabel('Count', fontsize=11, fontweight='bold')
    axes[0, 1].set_title('Transaction Types Distribution', fontsize=12, fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Quantity distribution in transactions
if 'quantite' in df_tempo_plot.columns:
    axes[1, 0].hist(df_tempo_plot['quantite'], bins=50, edgecolor='black', color='#FF6B6B', alpha=0.7)
    axes[1, 0].set_xlabel('Quantity', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Transaction Quantity Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].axvline(df_tempo_plot['quantite'].mean(), color='darkred', linestyle='--', linewidth=2, label=f"Mean: {df_tempo_plot['quantite'].mean():.0f}")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

# 4. Top products in transactions
if 'nom_produit' in df_tempo_plot.columns and 'quantite' in df_tempo_plot.columns:
    top_products_tempo = df_tempo_plot.groupby('nom_produit')['quantite'].sum().nlargest(10).sort_values()
    axes[1, 1].barh(range(len(top_products_tempo)), top_products_tempo.values, color='#95DAC1', alpha=0.7)
    axes[1, 1].set_yticks(range(len(top_products_tempo)))
    axes[1, 1].set_yticklabels([name[:25] + '...' if len(name) > 25 else name for name in top_products_tempo.index], fontsize=9)
    axes[1, 1].set_xlabel('Total Quantity', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Top 10 Products in Transactions', fontsize=12, fontweight='bold')
    axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Tempo Data Summary:")
print(f"   Total transaction records: {len(df_tempo):,}")
if 'date' in df_tempo_plot.columns:
    print(f"   Date range: {df_tempo_plot['date'].min().date()} to {df_tempo_plot['date'].max().date()}")
if 'type_transaction' in df_tempo_plot.columns:
    print(f"   Transaction types: {df_tempo_plot['type_transaction'].nunique()}")
if 'quantite' in df_tempo_plot.columns:
    print(f"   Average transaction quantity: {df_tempo_plot['quantite'].mean():.2f}")

---

## Step 4: Filter and Clean Transaction Data

Filter to keep only DELIVERY transactions and relevant columns for time series forecasting.

In [5]:
import pandas as pd

# Load tempo_ts.csv from data/raw directory
df_tempo = pd.read_csv('../data/raw/tempo_ts.csv')

print(f"Original shape: {df_tempo.shape}")
print(f"Original columns: {list(df_tempo.columns)}")

# Filter rows: keep only DELIVERY transactions
print("\n" + "="*70)
print("Filtering for type_transaction = 'DELIVERY'")
print("="*70)

if 'type_transaction' in df_tempo.columns:
    print(f"Unique transaction types: {df_tempo['type_transaction'].unique()}")
    df_tempo = df_tempo[df_tempo['type_transaction'] == 'DELIVERY']
    print(f"Shape after filtering for DELIVERY: {df_tempo.shape}")
else:
    print("Warning: 'type_transaction' column not found!")

# Keep only specific columns
columns_to_keep = [
    'id_produit',
    'cree_le',
    'categorie',
    'colisage fardeau',
    'colisage palette',
    'volume pcs (m3)',
    'Is_Gerbable',
    'quantite'
]

# Filter to keep only these columns
df_filtered = df_tempo[columns_to_keep]

print(f"\nFiltered shape: {df_filtered.shape}")
print(f"Filtered columns: {list(df_filtered.columns)}")

print("\nFirst 5 rows:")
print(df_filtered.head())

# Save to data/raw directory
df_filtered.to_csv('../data/raw/tempo_ts_grouped.csv', index=False)
print("\n✓ Filtered data saved to ../data/raw/tempo_ts_grouped.csv")

C:\Users\Afaf\AppData\Local\Temp\ipykernel_24044\3376754475.py:4: DtypeWarning: Columns (7,8,9,10,12,13,14,15,18,19,20,21,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tempo = pd.read_csv('../data/raw/tempo_ts.csv')


Original shape: (76890, 24)
Original columns: ['id_transaction', 'type_transaction', 'reference_transaction', 'cree_le', 'cree_par_id_utilisateur', 'statut', 'notes', 'no_ligne', 'id_produit', 'quantite', 'id_emplacement_source', 'id_emplacement_destination', 'lot_serie', 'code_motif', 'sku', 'nom_produit', 'unite_mesure', 'categorie', 'actif', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Poids(kg)', 'Is_Gerbable']

Filtering for type_transaction = 'DELIVERY'
Unique transaction types: ['texte' 'O' 'DELIVERY' 'RECEIPT']
Shape after filtering for DELIVERY: (67825, 24)

Filtered shape: (67825, 8)
Filtered columns: ['id_produit', 'cree_le', 'categorie', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Is_Gerbable', 'quantite']

First 5 rows:
  id_produit              cree_le        categorie colisage fardeau  \
2      31851  2025-03-12 00:00:00           MODULE               80   
3      31501  2025-03-12 00:00:00  DISPINA METALIC              120   
4      31501 

In [ ]:
# 📊 VISUALIZE: DELIVERY Transactions (Final Grouped Data)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Convert date to datetime
df_vis = df_filtered.copy()
if 'date' in df_vis.columns:
    df_vis['date'] = pd.to_datetime(df_vis['date'])
    
    # 1. Daily DELIVERY volume
    daily_delivery = df_vis.groupby(df_vis['date'].dt.date)['quantite'].sum()
    axes[0, 0].plot(daily_delivery.index, daily_delivery.values, marker='o', linewidth=2.5, markersize=5, color='#FF6B6B')
    axes[0, 0].set_xlabel('Date', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Total Delivery Quantity', fontsize=11, fontweight='bold')
    axes[0, 0].set_title('Daily DELIVERY Volume Over Time', fontsize=12, fontweight='bold')
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].fill_between(daily_delivery.index, daily_delivery.values, alpha=0.3, color='#FF6B6B')

    # 2. Deliveries by day of week
    df_vis['day_of_week'] = df_vis['date'].dt.day_name()
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_delivery = df_vis.groupby('day_of_week')['quantite'].sum().reindex(day_order, fill_value=0)
    colors_day = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3', '#A8E6CF']
    axes[0, 1].bar(range(len(day_delivery)), day_delivery.values, color=colors_day, alpha=0.7)
    axes[0, 1].set_xticks(range(len(day_delivery)))
    axes[0, 1].set_xticklabels([day[:3] for day in day_order], fontsize=10)
    axes[0, 1].set_ylabel('Total Quantity', fontsize=11, fontweight='bold')
    axes[0, 1].set_title('Delivery Volume by Day of Week', fontsize=12, fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Top 12 products in deliveries
top_delivery_products = df_vis.groupby('nom_produit')['quantite'].sum().nlargest(12).sort_values()
axes[1, 0].barh(range(len(top_delivery_products)), top_delivery_products.values, color='#4ECDC4', alpha=0.7)
axes[1, 0].set_yticks(range(len(top_delivery_products)))
axes[1, 0].set_yticklabels([name[:28] + '...' if len(name) > 28 else name for name in top_delivery_products.index], fontsize=9)
axes[1, 0].set_xlabel('Total Delivered Quantity', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Top 12 Products in DELIVERY Transactions', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Delivery quantity distribution
axes[1, 1].hist(df_vis['quantite'], bins=50, edgecolor='black', color='#95DAC1', alpha=0.7)
axes[1, 1].set_xlabel('Delivery Quantity', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Distribution of DELIVERY Quantities', fontsize=12, fontweight='bold')
mean_qty = df_vis['quantite'].mean()
axes[1, 1].axvline(mean_qty, color='darkgreen', linestyle='--', linewidth=2, label=f"Mean: {mean_qty:.0f}")
axes[1, 1].axvline(df_vis['quantite'].median(), color='orange', linestyle='--', linewidth=2, label=f"Median: {df_vis['quantite'].median():.0f}")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 DELIVERY Data Summary:")
print(f"   Total DELIVERY records: {len(df_filtered):,}")
if 'date' in df_vis.columns:
    print(f"   Date range: {df_vis['date'].min().date()} to {df_vis['date'].max().date()}")
    print(f"   Number of days: {df_vis['date'].nunique()}")
print(f"   Unique products delivered: {df_vis['id_produit'].nunique()}")
print(f"   Total quantity delivered: {df_vis['quantite'].sum():,.0f}")
print(f"   Average delivery quantity: {df_vis['quantite'].mean():.2f}")
print(f"   Median delivery quantity: {df_vis['quantite'].median():.0f}")

# Show which day has most deliveries
if 'day_of_week' in df_vis.columns:
    busiest_day = day_delivery.idxmax()
    print(f"   Busiest day: {busiest_day} ({day_delivery[busiest_day]:.0f} units)")

---

## ✅ Data Exploration Complete!

**Created Files:**
- `../data/raw/products_for_ts.csv` - Product demand history
- `../data/raw/tempo_ts.csv` - Full transaction data
- `../data/raw/tempo_ts_grouped.csv` - Filtered DELIVERY transactions

**Next Step:** Open `02_preprocessing.ipynb` to clean and prepare this data for modeling.